In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from sklearn import metrics
from time import time

%matplotlib inline

from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (confusion_matrix, accuracy_score)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score

In [ ]:
df = pd.read_excel('data/Swan Consulting 1 - Project Data.xlsx')

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=['Count', 'State', 'Country', 'CustomerID'])
df_clean = df_clean.drop(columns= ['Lat Long', 'City','Zip Code'])
df_clean = df_clean.drop(columns= ['Churn Label'])
df_clean = df_clean.drop(columns= ['Churn Reason'])
df_clean = df_clean.drop(columns= ['Latitude', 'Longitude'])

In [ ]:
df_clean['Gender'] = df_clean['Gender'].map({'Male':0, 'Female':1})
df_clean['Senior Citizen'] = df_clean['Senior Citizen'].map({'No':0, 'Yes':1})
df_clean['Partner'] = df_clean['Partner'].map({'No':0, 'Yes':1})
df_clean['Dependents'] = df_clean['Dependents'].map({'No':0, 'Yes':1})
df_clean['Phone Service'] = df_clean['Phone Service'].map({'No':0, 'Yes':1})
df_clean['Paperless Billing'] = df_clean['Paperless Billing'].map({'No':0, 'Yes':1})

In [ ]:
df_clean = pd.get_dummies(df_clean, columns=['Multiple Lines'], drop_first=True)
df_clean = pd.get_dummies(df_clean, columns=[
    'Internet Service',
    'Online Security',
    'Online Backup',
    'Device Protection',
    'Tech Support',
    'Streaming TV',
    'Streaming Movies',
    'Contract',
    'Payment Method'
], drop_first=True)

In [ ]:
df_clean = df_clean.astype({col: int for col in df_clean.select_dtypes(include='bool').columns})
df_clean[pd.to_numeric(df_clean['Total Charges'], errors='coerce').isna()][['Total Charges', 'Tenure Months']]
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce').fillna(0)

In [ ]:
X = df_clean.drop(columns=['Churn Value'])
y = df_clean['Churn Value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
tree_clf = DecisionTreeClassifier(random_state=42)

tree_params = {
    'max_depth': [3, 5, 7, 9, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}


In [ ]:
gs = GridSearchCV(estimator=tree_clf, 
                  param_grid=tree_params, 
                  cv=5, 
                  n_jobs=-1, 
                  verbose=2)

gs.fit(X_train, y_train)

In [ ]:
print("Best parameters found: ", gs.best_params_)

In [ ]:
best_tree = gs.best_estimator_
y_pred = best_tree.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))